# 공공데이터포털 오픈 API 실습 - 식품안전나라 개별기준규격 조회

> [식품안전나라 오픈API 상세 - 개별기준규격(I2580)](https://www.foodsafetykorea.go.kr/api/newDatasetDetail.do?menu_no=661&menu_grp=MENU_GRP31&p_svcTypeCd=API_TYPE06&svc_no=I2580)

`.env` 파일에 저장한 인증키로 식품안전나라 오픈 API를 직접 호출해보는 실습입니다.

> 인증키 발급 방법은 [`공공데이터활용 - 인증키 생성.md`](./공공데이터활용%20-%20인증키%20생성.md) 문서를 참고하세요.

## 0. 준비하기

1. 이 폴더의 `.env.sample`을 복사해 `.env` 파일을 만든다.
2. 발급받은 인증키를 `FOOD_SAFETY_API_KEY` 값으로 입력한다.

```
FOOD_SAFETY_API_KEY=발급받은_인증키
```

> 인증키가 아직 없다면 `sample`이라는 테스트용 키워드로 호출해볼 수 있습니다. 다만 이 실습에서 사용하는 개별기준규격(I2580)은 `sample` 키를 지원하지 않아 아래 실습을 정상적으로 마치려면 정식 인증키가 필요합니다.

In [1]:
import os
import json

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("FOOD_SAFETY_API_KEY", "sample")
API_KEY

'd831e885e9a04ccd8f72'

## 1. 요청 URL 구조 알아보기

API 문서에서 확인한 요청 주소 형식은 다음과 같습니다.

```
http://openapi.foodsafetykorea.go.kr/api/{인증키}/{서비스ID}/{요청타입}/{시작위치}/{종료위치}
```

| 순서 | 이름 | 설명 |
| --- | --- | --- |
| 1 | 인증키 | 발급받은 API Key (또는 테스트용 `sample`) |
| 2 | 서비스ID | 데이터셋을 구분하는 코드 (예: `I2580` = 개별기준규격) |
| 3 | 요청타입 | 응답 형식 (`json` 또는 `xml`) |
| 4 | 시작위치 | 조회를 시작할 행 번호 (1부터 시작) |
| 5 | 종료위치 | 조회를 마칠 행 번호 (한 번에 최대 1000건) |

In [2]:
BASE_URL = "http://openapi.foodsafetykorea.go.kr/api"
SERVICE_ID = "I2580"  # 개별기준규격
DATA_TYPE = "json"
START_IDX = 1
END_IDX = 5

url = f"{BASE_URL}/{API_KEY}/{SERVICE_ID}/{DATA_TYPE}/{START_IDX}/{END_IDX}"
url

'http://openapi.foodsafetykorea.go.kr/api/d831e885e9a04ccd8f72/I2580/json/1/5'

## 2. API 호출하고 응답 확인하기

In [9]:
response = requests.get(url)
print("status code:", response.status_code)

data = response.json()
print(json.dumps(data, ensure_ascii=False, indent=2)[:1000])

status code: 200
{
  "I2580": {
    "total_count": "0",
    "RESULT": {
      "MSG": "서버오류입니다.",
      "CODE": "ERROR-500"
    }
  }
}


### 응답 구조 이해하기

응답은 `{서비스ID: {...}}` 형태의 딕셔너리이며, 그 안에 다음 항목이 들어 있습니다.

| 키 | 설명 |
| --- | --- |
| `total_count` | 전체 데이터 건수 |
| `row` | 실제 데이터 목록 (정상일 때만 존재) |
| `RESULT.CODE` | 처리 결과 코드 |
| `RESULT.MSG` | 처리 결과 메시지 |

In [4]:
result = data[SERVICE_ID]["RESULT"]
print(result["CODE"], "-", result["MSG"])

ERROR-500 - 서버오류입니다.


> 인증키가 없거나 `sample`을 사용한 경우 위 코드가 `ERROR-100`(인증키 오류) 또는 `ERROR-500`(서버 오류)으로 나올 수 있습니다. 개별기준규격(I2580)은 정식으로 발급받은 인증키가 있어야 정상 응답(`INFO-000`)을 받을 수 있습니다.

## 3. 응답을 데이터프레임으로 변환하기

In [5]:
def call_food_api(service_id, api_key=API_KEY, data_type="json", start_idx=1, end_idx=5, **params):
    """식품안전나라 오픈 API를 호출하고 (결과코드, 데이터프레임)을 반환한다."""
    url = f"{BASE_URL}/{api_key}/{service_id}/{data_type}/{start_idx}/{end_idx}"
    if params:
        query = "&".join(f"{key}={value}" for key, value in params.items())
        url += f"/{query}"

    response = requests.get(url)
    body = response.json()[service_id]
    result = body["RESULT"]

    if result["CODE"] != "INFO-000":
        print(f"[{result['CODE']}] {result['MSG']}")
        return result["CODE"], pd.DataFrame()

    return result["CODE"], pd.DataFrame(body["row"])


code_, df_spec = call_food_api(SERVICE_ID, end_idx=20)
df_spec.head()

[ERROR-500] 서버오류입니다.


""


In [6]:
columns = ["PRDLST_CD", "PRDLST_CD_NM", "TESTITM_NM", "SPEC_VAL", "UNIT_NM", "VALD_BEGN_DT"]
df_spec[columns] if not df_spec.empty else df_spec

""


## 4. 조건을 추가해서 검색하기

개별기준규격 API는 `PRDLST_CD`(품목분류코드), `LAST_UPDT_DTM`(최종수정일) 조건을 추가로 지원합니다.

앞서 조회한 결과에서 품목분류코드를 하나 골라, 해당 품목의 기준규격만 다시 조회해봅니다.

In [7]:
if not df_spec.empty:
    target_code = df_spec.loc[0, "PRDLST_CD"]
    target_name = df_spec.loc[0, "PRDLST_CD_NM"]
    print(f"조회할 품목: {target_name} ({target_code})")

    code_, df_target = call_food_api(SERVICE_ID, end_idx=50, PRDLST_CD=target_code)
    df_target[columns]

## 5. 에러 코드 확인하기

자주 만나는 결과 코드는 다음과 같습니다.

| 코드 | 의미 |
| --- | --- |
| `INFO-000` | 정상 처리 |
| `INFO-200` | 해당 조건의 데이터 없음 |
| `ERROR-100` | 인증키가 없거나 유효하지 않음 |
| `ERROR-300` | 필수 요청 파라미터 누락 |
| `ERROR-336` | 데이터 요청 범위 초과 (최대 1000건) |
| `ERROR-500` | 서버 오류 (요청 형식·서비스ID 오류 포함) |

> 전체 오류 코드는 [식품안전나라 오픈API 가이드](https://www.foodsafetykorea.go.kr/api/howToUseApi.do?menu_grp=MENU_GRP34&menu_no=687)에서 확인할 수 있습니다.

## 6. 저장
조회한 결과를 `data/individual_standard_spec.csv`로 저장해보기

In [8]:
os.makedirs("data", exist_ok=True)
if not df_spec.empty:
    df_spec.to_csv("data/individual_standard_spec.csv", index=False, encoding="utf-8-sig")
    print("저장 완료")